# ComfyUI + MiniMax H3（Colab · v2）

**每天重置的只有 `/content` 本地盘**，Drive 和 Hugging Face 都不会丢。所以这版按下面的分工来做：

| 资源 | 放哪 | 为什么 |
|---|---|---|
| 小模型（SD1.5 2 GB、LoRA、VAE） | Google Drive（可选） | 有就拷到本地盘，30–90 s，不用重下 |
| 大模型（H3 最小组合 37 GB） | Hugging Face 流式拉 | 走 CDN 100+ MB/s，6–10 min 落地本地盘，比 Drive 的 FUSE 直接读快得多 |
| 产出（图/视频） | 想留就写回 Drive | Drive 是唯一能双向读写的 |

**Drive 是可选的**：挂载失败也能跑（模型会自动从 Hugging Face 拉）。

### 显存门槛

- MiniMax H3：≥ 24 GB → Colab **L4 / A100**（Pro 起）
- SDXL：12 GB → T4 勉强
- SD1.5：4 GB → T4 轻松

### 每天开机三步

1. 跑第 1 格（环境 + 可选挂 Drive）
2. 跑第 2 格（装 ComfyUI + 备模型，`WITH_H3=1` 才拉 37 GB）
3. 跑第 3 格（启动 + 公网地址）


In [ ]:
# ---------- 第 1 格：环境检查 + 可选挂载 Drive ----------
import os, shutil, subprocess

sm = shutil.which("nvidia-smi")
gpu = ""
if sm:
    gpu = subprocess.run([sm, "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip()
print("GPU:", gpu or "!! 没有 GPU：代码执行程序 → 更改运行时类型 → 选 T4/L4/A100")

# Drive 是可选的：要长期保存产出、或想优先从 Drive 拷模型，就取消下面两行注释并完成授权
# from google.colab import drive
# drive.mount("/content/drive")

DRIVE_OK = os.path.isdir("/content/drive/MyDrive")
print("DRIVE_OK =", DRIVE_OK, "(False 也能正常跑：模型会从 Hugging Face 拉)")

In [ ]:
# ---------- 第 2 格：装 ComfyUI + 备模型 ----------
WITH_H3 = "0"   # 改成 "1" 会额外拉 37 GB 的 H3 组合（需要 24 GB 显存 + ~60 GB 本地盘）
SYNC_DRIVE = "0"  # "1" = 下载完回写一份到 Drive，下次开机就不用再下

if not os.path.isdir("/content/ComfyUI/.git"):
    !git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI
!pip install -q -r /content/ComfyUI/requirements.txt

!curl -fsSL https://raw.githubusercontent.com/zhangzhen-Smile/comfyui-colab/main/setup_models.sh -o /tmp/setup_models.sh
!MODELS=/content/ComfyUI/models WITH_H3={WITH_H3} SYNC_DRIVE={SYNC_DRIVE} bash /tmp/setup_models.sh


In [ ]:
# ---------- 第 3 格：启动 ComfyUI + 公网隧道 ----------
import os, re, subprocess, time
import torch

LOG, TLOG = "/tmp/comfy.log", "/tmp/tunnel.log"
os.chdir("/content/ComfyUI")
open(LOG, "w").close()

args = ["python", "main.py", "--listen", "0.0.0.0", "--port", "8188", "--enable-cors-header", "*"]
if not torch.cuda.is_available():
    # torch 是 CPU 版、或运行时没挂 GPU 时必须加 --cpu，否则报 Torch not compiled with CUDA
    args.append("--cpu")
    print("!! 没有可用 CUDA：这次以 --cpu 启动（出图很慢）。想要 GPU 就改运行时类型后重跑本格。")
print("launch:", " ".join(args))

subprocess.Popen(args, stdout=open(LOG, "w"), stderr=subprocess.STDOUT)

for _ in range(60):
    time.sleep(2)
    if "To see the GUI go to" in open(LOG, errors="ignore").read():
        break
print(open(LOG, errors="ignore").read()[-1200:])

if not os.path.exists("/usr/local/bin/cloudflared"):
    subprocess.run(["wget", "-q",
                    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
                    "-O", "/usr/local/bin/cloudflared"], check=True)
    os.chmod("/usr/local/bin/cloudflared", 0o755)

open(TLOG, "w").close()
subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8188", "--no-autoupdate"],
                 stdout=open(TLOG, "w"), stderr=subprocess.STDOUT)

url = None
for _ in range(45):
    time.sleep(2)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open(TLOG, errors="ignore").read())
    if m:
        url = m.group(0)
        break
print("\nComfyUI 公网地址:", url or "(没抓到，看 /tmp/tunnel.log)")
print("断线后重跑这一格即可，模型不用重下。")

In [ ]:
# ---------- 第 4 格：把生成的作品推到 GitHub ----------
import os
from google.colab import userdata

# 只需做一次：左侧 🔑 Secrets 里加 GH_TOKEN = 你的 GitHub token（勾选 Notebook access）
try:
    os.environ["GITHUB_TOKEN"] = userdata.get("GH_TOKEN")
except Exception as e:
    print("还没配置 GH_TOKEN：", e)

!curl -fsSL https://raw.githubusercontent.com/zhangzhen-Smile/comfyui-colab/main/push_assets.sh -o /tmp/push_assets.sh
!REPO=zhangzhen-Smile/comfyui-assets bash /tmp/push_assets.sh /content/ComfyUI/output
